In [39]:
import pandas as pd
import re
import string
import nltk
import emoji
from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

In [2]:
df = pd.read_csv('datasets/IMDB Dataset.csv')

In [3]:
df.head(3)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [4]:
# check class balance
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [5]:
# check missing values
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
# check duplicates
df.duplicated().sum()

np.int64(418)

In [7]:
# drop duplicates
df.drop_duplicates(inplace=True)

In [8]:
# check duplicates again
df.duplicated().sum()

np.int64(0)

In [9]:
# convert text to lowercase
df['review'] = df['review'].str.lower()

In [10]:
# remove html tags
def remove_html(text):
    pattern = re.compile(r'<.*?>')
    return pattern.sub('', text)

In [11]:
df['review'] = df['review'].apply(remove_html)

In [12]:
df.head(3)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive


In [13]:
# remove URLs
def remove_url(text):
    pattern = re.compile(r'http\S+|www\.\S+')
    return pattern.sub('', text)

In [14]:
df['review'] = df['review'].apply(remove_url)

In [15]:
df.head(3)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive


In [16]:
# remove multiple spaces
def remove_multiple_spaces(text):
    pattern = re.compile(r'\s+')
    return pattern.sub(' ', text)

In [17]:
df['review'] = df['review'].apply(remove_multiple_spaces)

In [18]:
df.head(3)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive


In [19]:
# remove punctuations
def remove_punctuations(text):
    return text.translate(str.maketrans('','',string.punctuation))

In [20]:
df['review'] = df['review'].apply(remove_punctuations)

In [21]:
df.head(3)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive


In [22]:
# handle emojis
tqdm.pandas(desc="Processing Data")
df['review'] = df['review'].progress_apply(emoji.demojize)

Processing Data:   0%|          | 0/49582 [00:00<?, ?it/s]

In [23]:
# remove stop words
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Grv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
from nltk.corpus import stopwords
stopwords = stopwords.words('english')
def remove_stopwords(text):
    filtered_words = [word for word in text.split() if word not in stopwords]
    return " ".join(filtered_words)

In [25]:
df['review'] = df['review'].progress_apply(remove_stopwords)

Processing Data:   0%|          | 0/49582 [00:00<?, ?it/s]

In [26]:
df.head(3)

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive


In [27]:
" ".join(stopwords)

"a about above after again against ain all am an and any are aren aren't as at be because been before being below between both but by can couldn couldn't d did didn didn't do does doesn doesn't doing don don't down during each few for from further had hadn hadn't has hasn hasn't have haven haven't having he he'd he'll her here hers herself he's him himself his how i i'd if i'll i'm in into is isn isn't it it'd it'll it's its itself i've just ll m ma me mightn mightn't more most mustn mustn't my myself needn needn't no nor not now o of off on once only or other our ours ourselves out over own re s same shan shan't she she'd she'll she's should shouldn shouldn't should've so some such t than that that'll the their theirs them themselves then there these they they'd they'll they're they've this those through to too under until up ve very was wasn wasn't we we'd we'll we're were weren weren't we've what when where which while who whom why will with won won't wouldn wouldn't y you you'd you

In [28]:
# X,y split
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [29]:
y.head(3)

0    positive
1    positive
2    positive
Name: sentiment, dtype: object

In [31]:
# label encode y
le = LabelEncoder()
y = pd.Series(le.fit_transform(y))

In [32]:
y.head(3)

0    1
1    1
2    1
dtype: int64

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [36]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((39665, 1), (9917, 1), (39665,), (9917,))

In [50]:
# BOW
cv = CountVectorizer(ngram_range=(1,1), min_df=5, max_df=0.95)
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

In [51]:
gnb = GaussianNB()
gnb.fit(X_train_bow, y_train)

,priors,None
,var_smoothing,1e-09


In [ ]:
y_pred = gnb.predict(X_test_bow)
accuracy = accuracy_score(y_test, y_pred)